# Strands Agents with Bedrock AgentCore Memory — FSI Edition

This lab demonstrates persistent memory for AI agents — enabling them to remember client preferences, risk profiles, and past interactions across sessions.

## Overview

In this lab, you will:
- Create an AgentCore Memory instance with multiple strategies
- Store FSI client conversations (risk appetite, migration plans)
- Retrieve memories across sessions
- Build a memory-enabled agent that personalizes responses

## Why Memory for FSI?

- **Client continuity** — Agent remembers: "Vanguard prefers ESG, moderate risk"
- **Handoff support** — New TAM gets full context without re-asking
- **Compliance** — Auditable record of advice given
- **Personalization** — Tailored recommendations based on history

## Prerequisites

In [ ]:
import os
#os.environ['AWS_ACCESS_KEY_ID'] = ''
#os.environ['AWS_SECRET_ACCESS_KEY'] = ''
#os.environ['AWS_SESSION_TOKEN'] = ''
#os.environ['AWS_REGION'] = ''

In [ ]:
#%pip install -q strands-agents strands-agents-tools bedrock-agentcore rich

In [1]:
import boto3
region = boto3.session.Session().region_name
NOVA_PRO_MODEL_ID = 'us.amazon.nova-pro-v1:0'
if region.startswith('eu'): NOVA_PRO_MODEL_ID = 'eu.amazon.nova-pro-v1:0'
elif region.startswith('ap'): NOVA_PRO_MODEL_ID = 'apac.amazon.nova-pro-v1:0'
print(f'Region: {region}, Model: {NOVA_PRO_MODEL_ID}')

Region: ap-southeast-2, Model: apac.amazon.nova-pro-v1:0


## Demonstrating the Memory Problem

Without persistent memory, every new session starts fresh:

In [2]:
from strands import Agent
from strands.models import BedrockModel

agent = Agent(
    model=BedrockModel(model_id=NOVA_PRO_MODEL_ID, max_tokens=4096),
    system_prompt='You are a financial advisor assistant.',
)

# Simulate: user told the agent about their client last week
# But in a new session, the agent has no memory of it
agent('What do you know about my client Vanguard and their risk preferences?')

I'm sorry, but I can't share specific details about individual clients or their risk preferences due to confidentiality and privacy policies. It's important to maintain the trust and security of all clients' information.

However, I can provide general information about risk preferences and how they might influence investment strategies:

### Understanding Risk Preferences

1. **Risk Tolerance**:
   - **Conservative**: Prefers low-risk investments with stable returns. Typically invests in bonds, savings accounts, and blue-chip stocks.
   - **Moderate**: Balances between risk and reward. May invest in a mix of stocks, bonds, and mutual funds.
   - **Aggressive**: Willing to take higher risks for potentially higher returns. Often invests in growth stocks, options, and other high-risk assets.

2. **Risk Capacity**:
   - This refers to the actual ability to take on risk, often influenced by financial situation, investment horizon, and other financial goals.

### Factors Influencing Risk Pr

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "I'm sorry, but I can't share specific details about individual clients or their risk preferences due to confidentiality and privacy policies. It's important to maintain the trust and security of all clients' information.\n\nHowever, I can provide general information about risk preferences and how they might influence investment strategies:\n\n### Understanding Risk Preferences\n\n1. **Risk Tolerance**:\n   - **Conservative**: Prefers low-risk investments with stable returns. Typically invests in bonds, savings accounts, and blue-chip stocks.\n   - **Moderate**: Balances between risk and reward. May invest in a mix of stocks, bonds, and mutual funds.\n   - **Aggressive**: Willing to take higher risks for potentially higher returns. Often invests in growth stocks, options, and other high-risk assets.\n\n2. **Risk Capacity**:\n   - This refers to the actual ability to take on risk, often influenced by 

## Creating AgentCore Memory

Let's create a memory instance with three strategies:
- **Summary** — Compresses conversations into key points
- **User Preferences** — Captures client preferences and patterns
- **Semantic Facts** — Extracts factual knowledge from conversations

⏱️ This takes approximately 3 minutes to provision.

In [ ]:
from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore.memory.constants import StrategyType
from botocore.exceptions import ClientError
import boto3

region = boto3.session.Session().region_name
memory_client = MemoryClient(region_name=region)
memory_name = 'FSIClientMemory'

try:
    memory = memory_client.create_memory_and_wait(
        name=memory_name,
        description='FSI client context memory for advisory agents',
        strategies=[
            {
                StrategyType.SUMMARY.value: {
                    'name': 'SessionSummarizer',
                    'namespaces': ['fsi-agent/summaries/{actorId}/{sessionId}']
                }
            },
            {
                StrategyType.USER_PREFERENCE.value: {
                    'name': 'ClientPreferences',
                    'description': 'Captures client risk preferences and requirements',
                    'namespaces': ['fsi-agent/preferences/{actorId}'],
                }
            },
            {
                StrategyType.SEMANTIC.value: {
                    'name': 'FactExtractor',
                    'description': 'Stores facts about clients from conversations',
                    'namespaces': ['fsi-agent/semantic/{actorId}/'],
                }
            },
        ],
        event_expiry_days=30,
    )
    memory_id = memory.get('id')
    print(f'✅ Memory created: {memory_id}')
except ClientError as e:
    if 'already exists' in str(e):
        memories = memory_client.list_memories()
        memory_id = next((m['id'] for m in memories if m['id'].startswith(memory_name)), None)
        print(f'✅ Using existing memory: {memory_id}')
    else:
        raise e


## Storing Client Conversations

Let's simulate conversations about FSI clients and store them in memory:

In [ ]:
# Store Vanguard client conversation
USER_ID = 'tam_zohaib'
SESSION_ID = 'vanguard-session-001'

memory_client.create_event(
    memory_id=memory_id,
    actor_id=USER_ID,
    session_id=SESSION_ID,
    messages=[
        ('My client Vanguard is a large super fund based in Australia. They manage over $200B in assets.', 'USER'),
        ('Noted. Vanguard Australia is a major superannuation fund with $200B+ AUM.', 'ASSISTANT'),
        ('They have a moderate risk appetite and prefer ESG-aligned investments. They are migrating their core trading platform to EKS in Q3 2026.', 'USER'),
        ('Understood. Key notes: moderate risk, ESG preference, EKS migration planned for Q3 2026.', 'ASSISTANT'),
        ('Their primary concern is latency - they need sub-10ms for trade execution in ap-southeast-2. The escalation contact is John Chen, CTO.', 'USER'),
        ('Critical requirements noted: sub-10ms latency in ap-southeast-2 for trading. Escalation: John Chen (CTO).', 'ASSISTANT'),
    ],
)
print('✅ Vanguard conversation stored')

# Store Afterpay client conversation
SESSION_ID = 'afterpay-session-001'

memory_client.create_event(
    memory_id=memory_id,
    actor_id=USER_ID,
    session_id=SESSION_ID,
    messages=[
        ('Afterpay processes millions of BNPL transactions daily. They need real-time fraud detection with less than 100ms response time.', 'USER'),
        ('Afterpay requirements: real-time fraud detection, sub-100ms latency, high transaction volume.', 'ASSISTANT'),
        ('They are concerned about ASIC regulatory changes to BNPL. Their preferred region is ap-southeast-2 with DR in us-west-2.', 'USER'),
        ('Noted: ASIC regulatory risk, primary region ap-southeast-2, DR in us-west-2.', 'ASSISTANT'),
    ],
)
print('✅ Afterpay conversation stored')
print('\nWait 20-30 seconds for memory processing...')


## Retrieving Memories

Let's query the memory to see what was extracted:

In [ ]:
import time
time.sleep(30)  # Wait for memory processing

# Retrieve memories
USER_ID = 'tam_zohaib'

memories = memory_client.retrieve_memory(
    memory_id=memory_id,
    actor_id=USER_ID,
    query='What do I know about my FSI clients?',
)

print('=== Retrieved Memories ===')
for item in memories:
    print(f"\nType: {item.get('type', 'unknown')}")
    print(f"Content: {str(item.get('content', ''))[:200]}")


## Building a Memory-Enabled Agent

Now let's create an agent that retrieves memory before responding:

In [ ]:
import time
time.sleep(30)  # Wait for memory processing

# Retrieve memories
USER_ID = 'tam_zohaib'

memories = memory_client.retrieve_memory(
    memory_id=memory_id,
    actor_id=USER_ID,
    query='What do I know about my FSI clients?',
)

print('=== Retrieved Memories ===')
for item in memories:
    print(f"\nType: {item.get('type', 'unknown')}")
    print(f"Content: {str(item.get('content', ''))[:200]}")


In [ ]:
# Test: Ask about Afterpay
memory_agent("What are Afterpay's requirements and regulatory concerns?")


## Cleanup (Optional)

In [ ]:
# Uncomment to clean up
# memory_client.delete_memory_and_wait(memory_id=memory_id)
# print('✅ Memory deleted')


## Summary

- ✅ Created AgentCore Memory with summary, preference, and semantic strategies
- ✅ Stored FSI client conversations (Vanguard, Afterpay)
- ✅ Retrieved extracted insights (preferences, facts, summaries)
- ✅ Built a memory-enabled agent that personalizes responses

### FSI Takeaways

| Capability | FSI Value |
|-----------|----------|
| Persistent memory | Client context survives across sessions |
| Preference extraction | Auto-captures risk appetite, region preferences |
| Semantic facts | Stores key dates, contacts, requirements |
| Cross-session continuity | TAM handoff without losing context |

## 🎉 Workshop Complete!

You've built an FSI agent that can:
- Calculate financial metrics (Lab 00)
- Execute dynamic analysis code (Lab 01)
- Browse regulatory websites (Lab 02)
- Deploy tools as managed services (Lab 04)
- Provide full audit trails (Lab 05)
- Remember client context across sessions (Lab 06)